In [6]:
import pandas as pd
import numpy as np

# Importamos y analizamos la estructura del df
df = pd.read_csv('ckd.csv')
print('Filas:', df.shape[0], '| Columnas:', df.shape[1])
df.head()

Filas: 400 | Columnas: 26


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [10]:
print(df.dtypes)
print()

# Convertimos 3 variables que deberían ser numéricas: 'pcv', 'wc', 'rc'
# Limpiamos '\t?' y cualquier espacio suelto en las culumnas que son numéricas y deben de ser floats

cols = ['pcv', 'wc', 'rc']

for col in cols:
    df[col] = df[col].astype(str).str.strip()  # quita los espacios
    df[col] = df[col].replace({'?': np.nan, 'nan': np.nan})
    df[col] = pd.to_numeric(df[col], errors = 'coerce') # Lo que no se puede convertir -> NaN

# Limpiamos los '\t' que vienen en 'ckd'
df['classification'] = df['classification'].astype(str).str.strip()

print('Categorías liego del fix 🤠:', df['classification'].unique())
print(df[cols].dtypes)

id                  int64
age               float64
bp                float64
sg                float64
al                float64
su                float64
rbc                   str
pc                    str
pcc                   str
ba                    str
bgr               float64
bu                float64
sc                float64
sod               float64
pot               float64
hemo              float64
pcv               float64
wc                float64
rc                float64
htn                   str
dm                    str
cad                   str
appet                 str
pe                    str
ane                   str
classification        str
dtype: object

Categorías liego del fix 🤠: <ArrowStringArray>
['ckd', 'notckd']
Length: 2, dtype: str
pcv    float64
wc     float64
rc     float64
dtype: object


In [11]:
# Seleccionamos solo las variables continuas

var_con = ['age','bp','sg','al','su','bgr','bu','sc','sod','pot','hemo','pcv','wc','rc']

datos_cont = df[var_con].copy()
datos_cont.head()

,age,bp,sg,al,su,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc
0,48.0,80.0,1.020,1.0,0.0,121.0,36.0,1.2,NaN,NaN,15.4,44.0,7800.0,5.2
1,7.0,50.0,1.020,4.0,0.0,NaN,18.0,0.8,NaN,NaN,11.3,38.0,6000.0,NaN
2,62.0,80.0,1.010,2.0,3.0,423.0,53.0,1.8,NaN,NaN,9.6,31.0,7500.0,NaN
3,48.0,70.0,1.005,4.0,0.0,117.0,56.0,3.8,111.0,2.5,11.2,32.0,6700.0,3.9
4,51.0,80.0,1.010,2.0,0.0,106.0,26.0,1.4,NaN,NaN,11.6,35.0,7300.0,4.6


In [14]:
# Revisamos los valores faltantes

faltantes = datos_cont.isna().sum().sort_values(ascending = False)
print(faltantes)
print(f'Filas completas (sin ningún NA) si se eliminan filas con al menos 1 faltante: ' f'{datos_cont.dropna().shape[0]} de {datos_cont.shape[0]}')

rc      131
wc      106
pot      88
sod      87
pcv      71
hemo     52
su       49
sg       47
al       46
bgr      44
bu       19
sc       17
bp       12
age       9
dtype: int64
Filas completas (sin ningún NA) si se eliminan filas con al menos 1 faltante: 203 de 400


In [16]:
'''
Decisión a documentar en el artículo:
Eliminé las filas con al menos un valor faltante entre las 14 variables continuas (`dropna()`), en vez de
imputar, lo que nos deja 203 de 400 filas (~51%), lo cual es suficiente para ajustar los modelos
gaussianos sin necesidad de imputar (evitando introducir sesgo de imputación).

Si más adelante se decide que 203 filas son pocas, se puede revisar esta decisión y probar imputación.
'''

datos_limpios = datos_cont.dropna().reset_index(drop = True)
print('Filas finales 🤠: ', datos_limpios.shape[0])
datos_limpios.describe()

Filas finales 🤠:  203


,age,bp,sg,al,su,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc
count,203.000000,203.000000,203.000000,203.000000,203.000000,203.000000,203.000000,203.000000,203.000000,203.000000,203.000000,203.000000,203.000000,203.000000
mean,51.857143,74.926108,1.019039,0.822660,0.359606,138.029557,53.256158,2.251232,138.709360,4.568966,13.344335,40.669951,8592.610837,4.788670
std,15.641538,11.534256,0.005645,1.359994,0.971751,75.476369,45.771964,2.944274,6.908114,3.085574,2.833150,8.936211,2927.252383,1.001345
min,6.000000,50.000000,1.005000,0.000000,0.000000,70.000000,10.000000,0.400000,111.000000,2.500000,3.100000,9.000000,3800.000000,2.100000
25%,42.000000,70.000000,1.015000,0.000000,0.000000,95.500000,26.000000,0.800000,135.000000,3.750000,11.300000,34.000000,6700.000000,4.050000
50%,55.000000,80.000000,1.020000,0.000000,0.000000,117.000000,40.000000,1.100000,139.000000,4.500000,13.900000,42.000000,8100.000000,4.800000
75%,63.000000,80.000000,1.025000,1.500000,0.000000,133.500000,51.500000,2.200000,142.500000,4.900000,15.500000,48.000000,9800.000000,5.500000
max,90.000000,110.000000,1.025000,4.000000,5.000000,490.000000,309.000000,15.200000,150.000000,47.000000,17.800000,54.000000,26400.000000,8.000000


In [17]:
# Exportamos el nuevo .csv limpio
datos_limpios.to_csv('ckd_clean.csv', index = False)
print('Datos ready 🤠!')

Datos ready 🤠!
